# Transfer Learning dengan ResNet-50

**Topik:** Arsitektur CNN Modern  
**Dataset:** CIFAR-10  

Notebook ini merupakan dokumentasi eksperimen. Setiap bagian teori diikuti implementasi yang relevan, kemudian hasilnya dianalisis berdasarkan metrik yang dihasilkan program. Karena waktu dan perangkat komputasi dapat berbeda, angka hasil tidak ditulis sebagai klaim tetap; kesimpulan harus merujuk pada tabel output notebook.


## BAB 1 — PENDAHULUAN

### 1.1 Latar belakang

Convolutional Neural Network (CNN) telah menjadi pendekatan utama untuk klasifikasi citra karena mampu mempelajari fitur secara bertahap: lapisan awal menangkap pola sederhana seperti tepi dan tekstur, sedangkan lapisan yang lebih dalam membentuk representasi yang lebih abstrak. Namun, melatih jaringan yang dalam dari awal membutuhkan data, waktu, dan sumber daya yang besar.

Transfer learning menawarkan solusi dengan memanfaatkan model yang sebelumnya telah dilatih pada dataset besar, kemudian menyesuaikan classifier atau sebagian jaringan tersebut untuk tugas baru. Dalam eksperimen ini, ResNet-50 yang telah dilatih pada ImageNet diterapkan pada CIFAR-10. Perbedaan karakteristik kedua dataset menjadi alasan pentingnya preprocessing serta pemilihan strategi pelatihan yang tepat.

### 1.2 Rumusan masalah

1. Bagaimana menerapkan ResNet-50 pretrained untuk klasifikasi CIFAR-10?  
2. Apa perbedaan hasil feature extraction dan fine-tuning?  
3. Bagaimana pengaruh learning rate terhadap akurasi, loss, waktu, dan kestabilan training?

### 1.3 Tujuan dan kontribusi

Eksperimen ini bertujuan membangun baseline yang dapat direproduksi, membandingkan tiga konfigurasi transfer learning, serta menunjukkan bahwa pemilihan hyperparameter perlu didukung oleh pengukuran—bukan hanya oleh asumsi bahwa model yang lebih kompleks pasti lebih baik.


## BAB 2 — TINJAUAN PUSTAKA

### 2.1 ResNet-50 dan residual learning

ResNet memperkenalkan blok residual yang mempelajari fungsi `F(x)` lalu menjumlahkannya dengan input melalui koneksi pintas: `y = F(x) + x`. Koneksi ini membantu gradient mengalir pada jaringan yang dalam dan mengurangi masalah degradasi performa saat kedalaman jaringan bertambah (He et al., 2016). ResNet-50 adalah varian dengan 50 lapisan yang telah dilatih pada ImageNet dalam implementasi Torchvision.

### 2.2 Transfer learning

Pada **feature extraction**, seluruh backbone dibekukan (`requires_grad=False`) dan hanya fully connected layer yang baru dilatih. Strategi ini murah dan sesuai ketika data target terbatas atau kemiripan domain cukup tinggi. Pada **fine-tuning**, sebagian layer akhir—dalam notebook ini `layer4`—dibuka. Representasi tingkat tinggi dapat menyesuaikan CIFAR-10, tetapi risiko overfitting dan kebutuhan komputasinya meningkat.

### 2.3 CIFAR-10 dan perbedaan domain

CIFAR-10 berisi 60.000 citra RGB berukuran 32×32 yang terbagi ke dalam 10 kelas. Citra ImageNet yang menjadi sumber pretraining memiliki karakteristik resolusi dan distribusi yang berbeda. Oleh karena itu citra CIFAR-10 diubah menjadi 224×224 dan dinormalisasi menggunakan mean serta standard deviation ImageNet. Resize ini menjaga kompatibilitas dengan model, tetapi tidak menghilangkan keterbatasan informasi asli pada citra 32×32.

### 2.4 Learning rate dan evaluasi

Learning rate menentukan besar langkah pembaruan parameter. Classifier baru biasanya memerlukan learning rate lebih besar, sedangkan backbone pretrained menggunakan learning rate lebih kecil agar representasi yang telah dipelajari tidak rusak secara drastis. Evaluasi menggunakan loss cross-entropy dan accuracy; classification report menambahkan precision, recall, serta macro F1 untuk melihat performa tiap kelas.


### 2.5 Strategi transfer learning

Transfer learning memanfaatkan representasi yang telah dipelajari dari dataset sumber untuk menyelesaikan masalah pada dataset target. Pendekatan ini bermanfaat ketika dataset target lebih kecil atau biaya pelatihan dari awal terlalu besar. Pada eksperimen ini, ResNet-50 menggunakan bobot pralatih ImageNet, kemudian lapisan klasifikasi disesuaikan agar menghasilkan prediksi untuk sepuluh kelas CIFAR-10.

**Feature extraction** membekukan seluruh backbone dan hanya melatih classifier baru. Strategi ini membutuhkan parameter trainable yang lebih sedikit dan biasanya lebih cepat. **Fine-tuning** membuka `layer4` agar fitur tingkat tinggi dapat menyesuaikan karakteristik CIFAR-10. Konsekuensinya adalah kebutuhan komputasi dan risiko overfitting yang lebih besar.

### 2.6 Pertimbangan preprocessing

CIFAR-10 memiliki resolusi asli 32×32, sedangkan ResNet-50 pralatih menerima citra berukuran 224×224 dengan tiga kanal warna. Oleh sebab itu, citra diubah ukurannya dan dinormalisasi menggunakan statistik ImageNet. Augmentasi pembalikan horizontal hanya diterapkan pada data training; data test diproses tanpa augmentasi agar pengukuran tetap konsisten.


### 2.5 Convolutional Neural Network untuk Klasifikasi Citra

CNN memproses citra melalui koneksi lokal dan berbagi bobot. Lapisan konvolusi menangkap pola visual dari tingkat rendah hingga tingkat tinggi, sedangkan fungsi aktivasi memperkenalkan non-linearitas. Pooling atau stride dapat digunakan untuk mengurangi resolusi spasial dan biaya komputasi. Pada ResNet-50, ekstraksi fitur dilakukan oleh rangkaian convolutional layer, batch normalization, ReLU, dan residual block.

### 2.6 Residual Block dan ResNet-50

Residual block mempelajari perubahan atau residual terhadap input, bukan keseluruhan pemetaan secara langsung. Secara umum, keluarannya dapat ditulis sebagai `y = F(x) + x`, dengan `F(x)` merupakan rangkaian transformasi konvolusi dan `x` adalah koneksi pintas. Koneksi ini membantu menjaga aliran gradien pada jaringan yang dalam dan mengurangi risiko degradasi performa.

ResNet-50 menggunakan bottleneck block yang terdiri atas konvolusi 1×1, 3×3, dan 1×1. Konvolusi 1×1 mengurangi atau mengembalikan jumlah kanal sehingga model tetap efisien. Bobot pralatih ResNet-50 pada notebook ini berasal dari ImageNet, kemudian classifier terakhir diganti dengan linear layer berukuran sepuluh kelas.

### 2.7 Feature Extraction dan Fine-tuning

Pada feature extraction, bobot backbone dibekukan dan hanya classifier yang diperbarui. Jumlah parameter trainable menjadi kecil sehingga kebutuhan memori dan waktu pelatihan lebih rendah. Namun, fitur yang digunakan tidak mengalami penyesuaian terhadap domain CIFAR-10.

Pada fine-tuning, sebagian backbone dibuka agar representasi tingkat tinggi dapat beradaptasi dengan data target. Notebook ini membuka `layer4` dan memberikan learning rate yang lebih kecil dibandingkan classifier. Perbedaan learning rate tersebut penting karena classifier baru belum memiliki representasi yang stabil, sedangkan backbone telah memuat pengetahuan dari ImageNet.

### 2.8 Perbedaan Domain ImageNet dan CIFAR-10

ImageNet dan CIFAR-10 berbeda dalam resolusi, jumlah kelas, variasi objek, dan distribusi citra. CIFAR-10 terdiri atas citra RGB berukuran 32×32 dalam sepuluh kelas, sedangkan ImageNet memiliki citra beresolusi lebih tinggi dan kategori yang jauh lebih beragam. Resize ke 224×224 diperlukan untuk kompatibilitas dengan ResNet-50, tetapi tidak menambahkan detail baru yang hilang pada citra asli.

Perbedaan domain tersebut menjelaskan mengapa bobot pralatih tidak selalu langsung optimal pada dataset target. Preprocessing, augmentasi, dan pemilihan strategi fine-tuning menjadi bagian penting agar transfer learning dapat memberikan manfaat.

### 2.9 Evaluasi dan Generalisasi

Akurasi menunjukkan proporsi prediksi yang benar, sedangkan cross-entropy loss mengukur kualitas probabilitas prediksi. Keduanya perlu dibaca bersama karena akurasi tidak selalu mencerminkan tingkat keyakinan model. Precision, recall, dan macro F1 pada classification report memberikan gambaran performa tiap kelas.

Perbedaan akurasi training dan test digunakan sebagai generalization gap. Gap yang besar dapat menunjukkan overfitting, sedangkan akurasi yang sama-sama rendah dapat mengindikasikan underfitting atau proses pelatihan yang belum cukup lama. Waktu pelatihan dan jumlah parameter trainable juga dicatat agar kualitas model dapat dibandingkan dengan efisiensinya.


## BAB 3 — METODOLOGI PENELITIAN

### 3.1 Desain eksperimen

Penelitian bersifat eksperimental-komparatif. Semua konfigurasi menggunakan dataset, batch size, optimizer Adam, jumlah epoch, dan seed yang sama. Variabel yang dibandingkan adalah strategi pembekuan backbone dan learning rate. Hasil dicatat per epoch agar proses konvergensi dapat diperiksa, bukan hanya hasil akhir.

Sel berikut menyiapkan dependensi, seed, transformasi, dan DataLoader. `RandomHorizontalFlip` hanya digunakan pada data training sebagai augmentasi ringan; data test tidak diacak agar evaluasi konsisten.

In [ ]:
%pip install -q torch torchvision scikit-learn matplotlib pandas


In [ ]:
import random, time, numpy as np, torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
weights=models.ResNet50_Weights.DEFAULT; mean,std=weights.transforms().mean,weights.transforms().std
train_tf=transforms.Compose([transforms.Resize((224,224)),transforms.RandomHorizontalFlip(),transforms.ToTensor(),transforms.Normalize(mean,std)])
test_tf=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),transforms.Normalize(mean,std)])
train_ds=datasets.CIFAR10('data',train=True,download=True,transform=train_tf); test_ds=datasets.CIFAR10('data',train=False,download=True,transform=test_tf)
train_loader=DataLoader(train_ds,batch_size=64,shuffle=True,num_workers=2); test_loader=DataLoader(test_ds,batch_size=128,shuffle=False,num_workers=2)
print(device,len(train_ds),len(test_ds))


### 3.2 Lingkungan eksperimen dan spesifikasi Google Colab

Eksperimen ini dirancang untuk dijalankan pada Google Colab dengan konfigurasi berikut:

| Komponen | Spesifikasi yang digunakan |
|---|---|
| Runtime | Google Colab, Python 3 |
| Akselerator | GPU NVIDIA Tesla T4 |
| Memori GPU | Hingga 16 GB VRAM, bergantung pada GPU yang tersedia |
| RAM sistem | Sekitar 12–13 GB atau sesuai runtime yang tersedia |
| Penyimpanan | Penyimpanan sementara Colab untuk dataset dan hasil eksperimen |
| Framework | PyTorch dan Torchvision |
| Dataset | CIFAR-10 |

Tesla T4 dipilih karena mendukung CUDA dan mempercepat pelatihan ResNet-50. Spesifikasi GPU, RAM, versi CUDA, dan versi library dapat berbeda antar-sesi Colab. Oleh karena itu, jalankan sel pemeriksaan runtime dan periksa nilai `device` sebelum memulai eksperimen. Jika GPU tidak tersedia, notebook tetap dapat dijalankan menggunakan CPU dengan waktu pelatihan yang lebih lama.


### 3.2.1 Protokol reproducibility dan evaluasi

Seed ditetapkan pada awal notebook agar pembagian batch dan inisialisasi proses dapat diulang dengan kondisi yang serupa. Setiap konfigurasi menggunakan dataset, jumlah epoch, optimizer, dan prosedur evaluasi yang sama. Perbedaan hasil kemudian diinterpretasikan berdasarkan strategi pembekuan backbone dan learning rate.

Metrik utama yang digunakan adalah akurasi dan cross-entropy loss. Akurasi menunjukkan proporsi prediksi yang benar, sedangkan loss mengukur tingkat keyakinan model terhadap prediksinya. Generalization gap dihitung sebagai selisih akurasi training dan test; nilai yang besar dapat mengindikasikan overfitting. Waktu pelatihan dan jumlah parameter trainable digunakan untuk melengkapi perbandingan efisiensi.


### 3.3 Arsitektur dan prosedur pelatihan

Classifier bawaan ResNet-50 diganti dengan linear layer ber-output 10 kelas. Untuk setiap batch, model menghasilkan logits, loss dihitung menggunakan cross-entropy, lalu optimizer memperbarui parameter yang trainable. Pada mode evaluasi, dropout/batch normalization dijalankan dalam mode inference dan gradient tidak diperlukan.

Tiga konfigurasi yang diuji:

| Konfigurasi | Backbone | Learning rate classifier | Learning rate `layer4` |
|---|---|---:|---:|
| Feature extraction | Dibekukan | 1e-3 | — |
| Fine-tuning 1 | `layer4` dilatih | 1e-3 | 1e-4 |
| Fine-tuning 2 | `layer4` dilatih | 1e-4 | 1e-5 |


In [ ]:
def make_model(strategy='feature'):
    m=models.resnet50(weights=weights)
    for p in m.parameters(): p.requires_grad=False
    if strategy=='finetune':
        for p in m.layer4.parameters(): p.requires_grad=True
    m.fc=nn.Linear(m.fc.in_features,10); return m.to(device)
def count_trainable(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
criterion=nn.CrossEntropyLoss()
def run_epoch(m,loader,opt=None):
    train=opt is not None; m.train(train); total=correct=loss_sum=0
    for x,y in loader:
        x,y=x.to(device),y.to(device)
        if train: opt.zero_grad()
        out=m(x); loss=criterion(out,y)
        if train: loss.backward(); opt.step()
        loss_sum+=loss.item()*len(y); correct+=(out.argmax(1)==y).sum().item(); total+=len(y)
    return loss_sum/total,correct/total


## BAB 4 — HASIL DAN PEMBAHASAN

Setiap konfigurasi menyimpan loss dan accuracy per epoch. Tabel ringkasan memuat performa akhir, generalization gap, waktu pelatihan, dan jumlah parameter trainable. Kurva loss digunakan untuk melihat konvergensi, sedangkan kurva accuracy memperlihatkan perkembangan kemampuan klasifikasi.

Interpretasi harus memperhatikan beberapa hal. Akurasi training yang tinggi tetapi akurasi test rendah menunjukkan kemungkinan overfitting. Loss test yang terus menurun dengan kenaikan akurasi yang kecil dapat menunjukkan model mulai jenuh. Sebaliknya, fluktuasi besar dapat mengindikasikan learning rate terlalu besar atau training belum stabil. Waktu training tidak boleh dibandingkan tanpa memperhatikan perangkat yang digunakan.

### 4.1 Catatan Penggunaan Google Colab

Notebook dapat dijalankan di Google Colab dengan memilih **Runtime → Change runtime type**. Gunakan GPU jika tersedia agar proses pelatihan ResNet-50 berlangsung lebih cepat. Jika GPU tidak tersedia, kode tetap dapat dijalankan menggunakan CPU, tetapi waktu komputasi akan meningkat.

Dataset CIFAR-10 diunduh otomatis ke folder `data/` menggunakan parameter `download=True`. Folder tersebut berada di penyimpanan sementara Colab dan dapat terhapus ketika runtime dihapus atau direset. Jalankan sel secara berurutan, kemudian periksa output `device` untuk memastikan akselerator yang digunakan sudah sesuai.

In [ ]:
# Pemeriksaan runtime Google Colab atau Jupyter lokal.
import sys, platform
print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
try:
    import google.colab
    print('Runtime: Google Colab')
except ImportError:
    print('Runtime: lokal/Jupyter')


In [ ]:
def experiment(name,strategy,backbone_lr=None,classifier_lr=1e-3,epochs=5):
    m=make_model(strategy); groups=[{'params':m.fc.parameters(),'lr':classifier_lr}]
    if strategy=='finetune': groups.insert(0,{'params':m.layer4.parameters(),'lr':backbone_lr})
    opt=torch.optim.Adam(groups); rows=[]; start=time.perf_counter()
    for e in range(1,epochs+1):
        tl,ta=run_epoch(m,train_loader,opt); vl,va=run_epoch(m,test_loader)
        rows.append({'experiment':name,'epoch':e,'train_loss':tl,'train_accuracy':ta,'test_loss':vl,'test_accuracy':va})
        print(name,e,f'train_acc={ta:.3f}',f'test_acc={va:.3f}')
    for r in rows: r.update(seconds=time.perf_counter()-start,trainable_parameters=count_trainable(m))
    return m,rows
configs=[('feature_lr1e-3','feature',None,1e-3),('finetune_lr1e-4','finetune',1e-4,1e-3),('finetune_lr1e-5','finetune',1e-5,1e-4)]
models_run={}; results=[]
for name,strategy,blr,clr in configs:
    models_run[name],rows=experiment(name,strategy,blr,clr,epochs=5); results.extend(rows)


### 4.2 Analisis Metrik

Pilih konfigurasi terbaik berdasarkan kombinasi test accuracy tinggi, test loss rendah, generalization gap yang wajar, dan kurva yang stabil. Jangan memilih model hanya karena training accuracy tertinggi. Classification report melengkapi accuracy dengan precision, recall, dan F1 per kelas sehingga kelas yang sulit dikenali dapat diidentifikasi.

Perlu dicatat bahwa notebook ini menggunakan test set untuk perbandingan eksploratif. Dalam laporan eksperimen yang lebih ketat, sebagian data training sebaiknya dipisahkan menjadi validation set untuk pemilihan hyperparameter, sedangkan test set hanya digunakan sekali pada evaluasi akhir.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import classification_report
df=pd.DataFrame(results); display(df)
summary=df.groupby('experiment').tail(1).copy(); summary['generalization_gap']=summary.train_accuracy-summary.test_accuracy
display(summary.sort_values('test_accuracy',ascending=False))
fig,ax=plt.subplots(1,2,figsize=(13,4))
for name,g in df.groupby('experiment'):
    ax[0].plot(g.epoch,g.test_loss,label=name); ax[1].plot(g.epoch,g.test_accuracy,label=name)
ax[0].set(title='Test loss',xlabel='Epoch'); ax[1].set(title='Test accuracy',xlabel='Epoch'); ax[0].legend(); ax[1].legend(); plt.show()
best=summary.sort_values('test_accuracy',ascending=False).iloc[0].experiment; m=models_run[best]; yt=[];yp=[];m.eval()
with torch.no_grad():
    for x,y in test_loader: yt.extend(y.numpy()); yp.extend(m(x.to(device)).argmax(1).cpu().numpy())
print('Best:',best); print(classification_report(yt,yp,target_names=train_ds.classes))


## BAB 5 — KESIMPULAN, KETERBATASAN, DAN SARAN

### 5.1 Kesimpulan

Feature extraction merupakan baseline yang efisien karena hanya classifier baru yang diperbarui. Fine-tuning memberi ruang bagi fitur tingkat tinggi untuk beradaptasi dengan CIFAR-10, tetapi menambah biaya komputasi dan memerlukan learning rate backbone yang lebih kecil. Kesimpulan numerik ditampilkan oleh sel kesimpulan otomatis berdasarkan tabel hasil aktual. Dengan demikian, konfigurasi terbaik, generalization gap, jumlah parameter, dan waktu pelatihan tidak perlu diisi secara manual.


In [ ]:
# Kesimpulan otomatis berdasarkan hasil eksperimen
summary_auto=summary.sort_values('test_accuracy', ascending=False).copy()
best_row=summary_auto.iloc[0]
print('KONFIGURASI TERBAIK:', best_row['experiment'])
print(f"Test accuracy: {best_row['test_accuracy']:.4f}")
print(f"Test loss: {best_row['test_loss']:.4f}")
print(f"Generalization gap: {best_row['generalization_gap']:.4f}")
print(f"Parameter trainable: {int(best_row['trainable_parameters']):,}")
print(f"Waktu training: {best_row['seconds']:.2f} detik")

if best_row['experiment'].startswith('feature'):
    strategy_note='Feature extraction memberikan hasil terbaik pada konfigurasi ini.'
else:
    strategy_note='Fine-tuning memberikan hasil terbaik pada konfigurasi ini.'
if best_row['generalization_gap'] > 0.10:
    generalization_note='Generalization gap relatif besar sehingga terdapat indikasi overfitting.'
else:
    generalization_note='Generalization gap masih relatif terkendali pada eksperimen ini.'
print(strategy_note)
print(generalization_note)
print('Kesimpulan ini dihasilkan dari tabel hasil aktual dan akan berubah jika notebook dijalankan dengan konfigurasi berbeda.')


### 5.2 Keterbatasan

Eksperimen ini belum menggunakan validation split, checkpoint model terbaik, scheduler learning rate, atau confusion matrix. Resize 32×32 ke 224×224 juga meningkatkan biaya komputasi tanpa menambah informasi detail asli. Hasil dapat berubah sesuai versi library, GPU/CPU, jumlah worker, dan kondisi randomisasi.

### 5.3 Saran pengembangan

Eksperimen lanjutan dapat menambahkan validation set, early stopping, scheduler, weight decay, top-k accuracy, confusion matrix, dan pengujian beberapa seed. Untuk laporan final, cantumkan spesifikasi perangkat, versi Python/PyTorch/Torchvision, waktu pelatihan, jumlah epoch, serta checkpoint model terbaik.

In [ ]:
# Validasi informasi keterbatasan eksperimen
print('VALIDASI KETERBATASAN EKSPERIMEN')
print(f'Jumlah data training : {len(train_ds):,}')
print(f'Jumlah data test     : {len(test_ds):,}')
print(f'Jumlah kelas         : {len(train_ds.classes)}')
print(f'Nama kelas           : {train_ds.classes}')

assert len(train_ds) == 50_000, 'Ukuran training CIFAR-10 tidak sesuai.'
assert len(test_ds) == 10_000, 'Ukuran test CIFAR-10 tidak sesuai.'
assert len(train_ds.classes) == 10, 'Jumlah kelas CIFAR-10 tidak sesuai.'

# CIFAR-10 asli berukuran 32x32 RGB sebelum transformasi Resize.
raw_image, raw_label = train_ds.data[0], train_ds.targets[0]
print(f'Ukuran citra asli    : {raw_image.shape[1]}x{raw_image.shape[0]} RGB')
assert raw_image.shape == (32, 32, 3), 'Resolusi citra asli tidak sesuai.'

# Notebook menggunakan test set secara langsung dan tidak membuat validation split terpisah.
print('Validation split     : Tidak tersedia; test set digunakan untuk evaluasi eksploratif.')
print(f'Perangkat evaluasi   : {device}')
print('Status               : Semua pemeriksaan dasar berhasil.')


### 5.3 Saran Pengembangan

Saran pengembangan berikut dihasilkan berdasarkan metrik aktual dan keterbatasan eksperimen yang terdeteksi oleh script validasi.


In [ ]:
# Saran otomatis berdasarkan hasil eksperimen
print('SARAN OTOMATIS')
max_gap=float(summary['generalization_gap'].max())
mean_gap=float(summary['generalization_gap'].mean())
if max_gap > 0.10:
    print('- Tambahkan validation split dan early stopping untuk mengurangi risiko overfitting.')
    print('- Pertimbangkan augmentasi data atau weight decay tambahan.')
else:
    print('- Generalization gap masih terkendali; ulangi eksperimen dengan beberapa random seed.')
if len(configs) < 4:
    print('- Tambahkan variasi learning rate dan scheduler untuk memperluas evaluasi hyperparameter.')
if not torch.cuda.is_available():
    print('- Gunakan GPU jika tersedia untuk mengurangi waktu pelatihan ResNet-50.')
else:
    print('- Catat tipe GPU dan waktu pelatihan agar hasil mudah direproduksi.')
print('- Tambahkan confusion matrix dan classification report pada validation set untuk evaluasi lanjutan.')
print(f'- Rata-rata generalization gap: {mean_gap:.4f}')


## REFERENSI

1. He, K., Zhang, X., Ren, S., dan Sun, J. (2016). *Deep Residual Learning for Image Recognition*. IEEE Conference on Computer Vision and Pattern Recognition, 770–778.
2. Pan, S. J., dan Yang, Q. (2010). *A Survey on Transfer Learning*. IEEE Transactions on Knowledge and Data Engineering, 22(10), 1345–1359.
3. Krizhevsky, A. (2009). *Learning Multiple Layers of Features from Tiny Images*. Technical Report, University of Toronto.
4. Deng, J., et al. (2009). *ImageNet: A Large-Scale Hierarchical Image Database*. IEEE Conference on Computer Vision and Pattern Recognition, 248–255.
5. Ioffe, S., dan Szegedy, C. (2015). *Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift*. Proceedings of ICML, 448–456.
6. Kingma, D. P., dan Ba, J. (2015). *Adam: A Method for Stochastic Optimization*. International Conference on Learning Representations.
7. Paszke, A., et al. (2019). *PyTorch: An Imperative Style, High-Performance Deep Learning Library*. Advances in Neural Information Processing Systems, 32.
8. Marcel, S., dan Rodriguez, Y. (2010). *Torchvision the Machine-Vision Package of Torch*. Proceedings of the 18th ACM International Conference on Multimedia, 1485–1488.
9. Shorten, C., dan Khoshgoftaar, T. M. (2019). *A Survey on Image Data Augmentation for Deep Learning*. Journal of Big Data, 6(1), 60.
10. Goodfellow, I., Bengio, Y., dan Courville, A. (2016). *Deep Learning*. MIT Press.
